# Railroad Interest Labor Coverage Regression

Tests whether newspaper owners with railroad financial ties produced more anti-labor coverage.

**Unit of observation:** newspaper Ã— year  
**Outcome:** `anti_labor_intensity` = anti-labor articles / total labor articles  
**Treatment:** `railroad_interest` = 1 if any coded owner had a financial railroad tie (stockholder, board member, company owner, donation, professional services, or family connection)

**Control group:** only persons with at least one explicit 0 across the railroad interest fields and no positive values â€” "confirmed non-railroad". Persons with all-NaN coding (never coded) are dropped from the analysis.

In [1]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

DB_PATH   = '../data/newspapers.db'
OE_PATH   = '../data/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/personnel_coding/combined_coding.csv'

## 1. Build railroad interest indicator

In [2]:
coding = pd.read_csv(CODE_PATH)

RAILROAD_COLS = [
    'railroad_stockholder', 'railroad_board_member', 'railroad_company_owner',
    'railroad_donation', 'railroad_professional_services', 'family_connection_railroad'
]

# railroad_interest=1: any column is > 0
coding['railroad_interest'] = (coding[RAILROAD_COLS].fillna(0) > 0).any(axis=1).astype(int)

# confirmed_control=1: at least one column has an explicit 0, and none are > 0
has_explicit_zero = (coding[RAILROAD_COLS] == 0).any(axis=1)
coding['confirmed_control'] = has_explicit_zero & (coding['railroad_interest'] == 0)

# Keep only persons who are either treated or confirmed controls
person_rr = coding[coding['railroad_interest'] | coding['confirmed_control']].copy()
person_rr = person_rr[['person_id', 'name', 'railroad_interest']].dropna(subset=['person_id'])
person_rr['person_id'] = person_rr['person_id'].astype(int)

print(f"Treated (railroad_interest=1): {coding['railroad_interest'].sum()}")
print(f"Confirmed controls:            {coding['confirmed_control'].sum()}")
print(f"Dropped (all NaN / uncoded):   {len(coding) - len(person_rr)}")
print(f"\nPersons in analysis: {len(person_rr)}")
print("\nTreated:")
for _, r in person_rr[person_rr['railroad_interest'] == 1].iterrows():
    print(f"  {r['name']}")
print("\nControls:")
for _, r in person_rr[person_rr['railroad_interest'] == 0].iterrows():
    print(f"  {r['name']}")

Treated (railroad_interest=1): 24
Confirmed controls:            29
Dropped (all NaN / uncoded):   410

Persons in analysis: 53

Treated:
  Whitelaw Reid
  Edgar Snowden
  James Gordon Bennett
  Lewis Baker
  Buckley B. Paddock | Walter Malone
  John F. Morse
  Michael Hahn
  John Bennett Carrington, Sr.
  J. H. Estill
  H. Estill
  H. Smith | A. J. Steinman | E. H. Thomas
  Charles M. Stone
  Edward Scull
  C. W. Willard
  Levi G. Gould
  Wm. H. Simpson
  Alfales Young
  George E. Lemon
  Ignatius Donnelly
  General John M. Hedrick | Major Augustus H. Hamilton
  John M. Hedrick
  D. R. Anthony
  Jay Gould
  Wilmer Atkinson | Howard Jenkins

Controls:
  Charles A. Dana | George Ripley | Bayard Taylor | Henry J. Raymond | Horace Greeley | Margaret Fuller
  Crosby Noyes
  Harlan P. Hall
  John C. New
  Berry R. Sulgrove
  Henry Rust Mighels
  Alfred Doten
  George Vernon
  J. W. Potter
  William Osman | Douglas Hapeman
  John M. Keating | Matthew Gallaway
  William L. Norris
  A. McGrego

## 2. Link owners â†’ newspapers (issn + active years)

In [3]:
oe = pd.read_csv(OE_PATH)

# Keep only rows with a known person and a valid issn
oe_persons = oe.dropna(subset=['person_id', 'issn']).copy()
oe_persons['person_id'] = oe_persons['person_id'].astype(int)

# Merge in railroad interest
oe_persons = oe_persons.merge(person_rr, on='person_id', how='inner')

# Expand year ranges: 'years' column is like "1871; 1873; 1876"
def parse_years(s):
    if pd.isna(s):
        return []
    return [int(y.strip()) for y in str(s).split(';') if y.strip().isdigit()]

rows = []
for _, row in oe_persons.iterrows():
    for yr in parse_years(row['years']):
        rows.append({'issn': row['issn'], 'year': yr,
                     'person_id': row['person_id'], 'railroad_interest': row['railroad_interest'],
                     'role': row['role']})

person_paper_year = pd.DataFrame(rows)
print(f"Person-paper-year rows: {len(person_paper_year)}")
print(person_paper_year.head())

Person-paper-year rows: 771
        issn  year  person_id  railroad_interest       role
0  1941-0646  1890          3                  1  publisher
1  1941-0646  1869          4                  0     editor
2  1941-0646  1871          4                  0     editor
3  1941-0646  1873          3                  1     editor
4  1941-0646  1876          3                  1     editor


In [4]:
# Aggregate to newspaper-year: 1 if ANY owner in that year had railroad interest
paper_year_rr = (
    person_paper_year
    .groupby(['issn', 'year'])
    .agg(
        railroad_interest=('railroad_interest', 'max'),
        has_editor=('role', lambda x: int((x == 'editor').any())),
        has_publisher=('role', lambda x: int((x == 'publisher').any())),
    )
    .reset_index()
)

print(f"Newspaper-years with owner coding: {len(paper_year_rr)}")
print(paper_year_rr['railroad_interest'].value_counts())
print(f"\nRole coverage:")
print(f"  has_editor=1:    {paper_year_rr['has_editor'].sum()}")
print(f"  has_publisher=1: {paper_year_rr['has_publisher'].sum()}")
print(f"  both:            {((paper_year_rr['has_editor']==1) & (paper_year_rr['has_publisher']==1)).sum()}")

Newspaper-years with owner coding: 494
railroad_interest
0    261
1    233
Name: count, dtype: int64

Role coverage:
  has_editor=1:    389
  has_publisher=1: 382
  both:            277


## 3. Compute anti-labor intensity from article_sentiment

In [5]:
conn = sqlite3.connect(DB_PATH)
sent = pd.read_sql("""
    SELECT issn, year, labor_sentiment
    FROM article_sentiment
    WHERE issn != '' AND labor_sentiment IS NOT NULL
""", conn)
conn.close()

# Count labor articles by sentiment class per newspaper-year
counts = (
    sent.groupby(['issn', 'year', 'labor_sentiment'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['anti_labor', 'pro_labor', 'neutral']:
    if col not in counts.columns:
        counts[col] = 0

counts['total_labor'] = counts['anti_labor'] + counts['pro_labor'] + counts['neutral']
counts['anti_labor_intensity'] = counts['anti_labor'] / counts['total_labor']

print(f"Newspaper-years with labor sentiment: {len(counts)}")
counts[['anti_labor', 'pro_labor', 'neutral', 'total_labor', 'anti_labor_intensity']].describe()

Newspaper-years with labor sentiment: 4216


labor_sentiment,anti_labor,pro_labor,neutral,total_labor,anti_labor_intensity
count,4216.000000,4216.000000,4216.000000,4216.000000,4216.000000
mean,6.636622,6.650380,6.936670,20.223672,0.308560
std,23.683407,19.409373,24.510574,64.194626,0.293107
min,0.000000,0.000000,0.000000,1.000000,0.000000
25%,0.000000,1.000000,0.000000,2.000000,0.000000
50%,1.000000,2.000000,1.000000,5.000000,0.272727
75%,4.000000,5.000000,4.000000,14.000000,0.500000
max,745.000000,461.000000,593.000000,1799.000000,1.000000


## 4. Merge and describe the analysis sample

In [6]:
df = counts.merge(paper_year_rr, on=['issn', 'year'], how='inner')

print(f"Analysis sample: {len(df)} newspaper-years, {df['issn'].nunique()} newspapers")
print(f"railroad_interest=1: {df['railroad_interest'].sum()} obs | railroad_interest=0: {(df['railroad_interest']==0).sum()} obs")
print()
print(df.groupby('railroad_interest')[['anti_labor_intensity', 'total_labor']].describe().T)

Analysis sample: 293 newspaper-years, 54 newspapers
railroad_interest=1: 121 obs | railroad_interest=0: 172 obs

railroad_interest                    0           1
anti_labor_intensity count  172.000000  121.000000
                     mean     0.288570    0.356390
                     std      0.216290    0.212560
                     min      0.000000    0.000000
                     25%      0.142857    0.232258
                     50%      0.285714    0.348837
                     75%      0.400000    0.448052
                     max      1.000000    1.000000
total_labor          count  172.000000  121.000000
                     mean    29.622093   65.900826
                     std     45.956911   96.430148
                     min      1.000000    1.000000
                     25%      5.000000    7.000000
                     50%     12.000000   25.000000
                     75%     31.000000   80.000000
                     max    338.000000  523.000000


## 5. Add control variables (state, political affiliation)

In [7]:
# Add state info
oe_meta = oe[['issn', 'state']].drop_duplicates('issn')
df = df.merge(oe_meta, on='issn', how='left')
df['year_str'] = df['year'].astype(str)  # for year fixed effects

print(f"States in sample: {df['state'].nunique()}")
print(df['state'].value_counts().head(10))

States in sample: 26
state
OHIO                    24
NEW YORK                19
DISTRICT OF COLUMBIA    16
NEVADA                  15
IOWA                    15
KANSAS                  14
VERMONT                 13
ILLINOIS                13
PENNSYLVANIA            13
VIRGINIA                13
Name: count, dtype: int64


In [8]:
# Load master.csv to get political affiliation (nearest Ayer directory year)
master_pol = pd.read_csv('../data/master.csv', low_memory=False)
pol_aff_cols = [c for c in master_pol.columns if 'political_affiliation' in c.lower()]

pol_long = []
for col in pol_aff_cols:
    yr = int(col.split()[0])
    tmp = master_pol[['issn', col]].dropna(subset=['issn', col]).copy()
    tmp = tmp.rename(columns={col: 'political_affiliation'})
    tmp['pol_year'] = yr
    pol_long.append(tmp)
pol_long = pd.concat(pol_long, ignore_index=True)

pol_lookup = {}
for _, row in pol_long.iterrows():
    pol_lookup.setdefault(row['issn'], []).append((row['pol_year'], row['political_affiliation']))

def nearest_pol_aff(issn, year):
    entries = pol_lookup.get(issn)
    if not entries:
        return None
    return min(entries, key=lambda x: abs(x[0] - year))[1]

def pol_to_republican(aff):
    if aff is None or (isinstance(aff, float)):
        return None
    aff_lower = str(aff).lower()
    if 'republican' in aff_lower:
        return 1
    elif 'democrat' in aff_lower:
        return 0
    return None

df['political_affiliation_raw'] = df.apply(lambda r: nearest_pol_aff(r['issn'], r['year']), axis=1)
df['republican'] = df['political_affiliation_raw'].map(pol_to_republican)

print(f"Papers with Republican affiliation : {(df['republican']==1).sum()} obs")
print(f"Papers with Democrat  affiliation  : {(df['republican']==0).sum()} obs")
print(f"Papers with other/unknown          : {df['republican'].isna().sum()} obs")


Papers with Republican affiliation : 86 obs
Papers with Democrat  affiliation  : 84 obs
Papers with other/unknown          : 123 obs


In [9]:
# Person-level political affiliation from combined_coding.csv
person_pol = coding[['person_id', 'political_affiliation']].dropna(subset=['person_id']).copy()
person_pol['person_id'] = person_pol['person_id'].astype(int)
person_pol = person_pol.rename(columns={'political_affiliation': 'person_political_affiliation'})

# Merge person politics into person_paper_year, then aggregate to newspaper-year
ppy_pol = person_paper_year.merge(person_pol, on='person_id', how='left')

def person_pol_to_republican(aff):
    if pd.isna(aff):
        return None
    aff_lower = str(aff).lower()
    if 'republican' in aff_lower:
        return 1
    elif 'democrat' in aff_lower:
        return 0
    return None

ppy_pol['person_republican'] = ppy_pol['person_political_affiliation'].map(person_pol_to_republican)

# Aggregate: take max (1 if any person is Republican, 0 if all are Democrat, NaN if unknown)
paper_year_person_pol = (
    ppy_pol.groupby(['issn', 'year'])['person_republican']
    .apply(lambda x: x.max() if x.notna().any() else None)
    .reset_index()
)

df = df.merge(paper_year_person_pol, on=['issn', 'year'], how='left')

print(f"Person-level Republican: {(df['person_republican']==1).sum()} obs")
print(f"Person-level Democrat:   {(df['person_republican']==0).sum()} obs")
print(f"Person-level unknown:    {df['person_republican'].isna().sum()} obs")

Person-level Republican: 137 obs
Person-level Democrat:   119 obs
Person-level unknown:    37 obs


In [10]:
# Compare newspaper-level vs person-level political affiliation
compare = df[['republican', 'person_republican']].dropna()
ct = pd.crosstab(
    compare['republican'].map({1: 'Paper: Republican', 0: 'Paper: Democrat'}),
    compare['person_republican'].map({1: 'Person: Republican', 0: 'Person: Democrat'}),
    margins=True
)
print("Newspaper-level vs person-level political affiliation:")
print(ct)
print(f"\nAgreement rate: {(compare['republican'] == compare['person_republican']).mean():.1%}")

Newspaper-level vs person-level political affiliation:
person_republican  Person: Democrat  Person: Republican  All
republican                                                  
Paper: Democrat                  74                   9   83
Paper: Republican                 0                  74   74
All                              74                  83  157

Agreement rate: 94.3%


## 6. Add county-level population from Census (1870, 1880)

In [11]:
import numpy as np

# Load city-county mapping and Census data (1870, 1880, 1890)
city_county = pd.read_csv('../data/Census/city_county_mapping.csv')
c70 = pd.read_csv('../data/Census/nhgis0001_ds16_1870_county.csv')
c80 = pd.read_csv('../data/Census/nhgis0001_ds22_1880_county.csv')
c90 = pd.read_csv('../data/Census/nhgis0002_ds26_1890_county.csv')

# Merge census years by county
c70r = c70.rename(columns={'STATE': 'census_state', 'COUNTY': 'county', 'AJR001': 'pop_1870'})
c80r = c80.rename(columns={'STATE': 'census_state', 'COUNTY': 'county', 'AOB001': 'pop_1880'})
c90r = c90.rename(columns={'STATE': 'census_state', 'COUNTY': 'county', 'ASW001': 'pop_1890'})
census = (
    c70r[['census_state', 'county', 'pop_1870']]
    .merge(c80r[['census_state', 'county', 'pop_1880']], on=['census_state', 'county'], how='outer')
    .merge(c90r[['census_state', 'county', 'pop_1890']], on=['census_state', 'county'], how='outer')
)

# Join mapping → census → issn
city_pop = city_county.merge(census, on=['census_state', 'county'], how='left')

master_full = pd.read_csv('../data/master.csv', low_memory=False, usecols=['issn', 'town', 'state', 'newspaper_name'])
master_geo = master_full.dropna(subset=['issn', 'town', 'state']).drop_duplicates('issn')
master_geo['town_upper'] = master_geo['town'].str.upper().str.strip()
master_geo['state_upper'] = master_geo['state'].str.upper().str.strip()
city_pop['town_upper'] = city_pop['town'].str.upper().str.strip()
city_pop['state_upper'] = city_pop['state'].str.upper().str.strip()

issn_pop = master_geo.merge(city_pop[['town_upper', 'state_upper', 'pop_1870', 'pop_1880', 'pop_1890']],
                            on=['town_upper', 'state_upper'], how='left')

# Diagnose unmatched analysis ISSNs
analysis_issns = set(df['issn'].unique())
matched_issns = set(issn_pop.dropna(subset=['pop_1870', 'pop_1880', 'pop_1890'], how='all')['issn'])
unmatched = analysis_issns - matched_issns

if unmatched:
    # Split: ISSNs missing from master_geo (no town/state) vs present but no county mapping
    no_geo = unmatched - set(master_geo['issn'])
    has_geo_no_match = unmatched & set(master_geo['issn'])

    print(f"WARNING: {len(unmatched)} analysis ISSNs have no Census match.\n")

    if no_geo:
        no_geo_info = master_full[master_full['issn'].isin(no_geo)][['issn', 'newspaper_name', 'town', 'state']].drop_duplicates('issn')
        if no_geo_info.empty:
            # ISSNs not in master.csv at all — look them up from oe
            oe_info = oe[oe['issn'].isin(no_geo)][['issn', 'newspaper_name', 'state']].drop_duplicates('issn')
            print(f"  {len(no_geo)} ISSNs missing town/state in master.csv (need manual entry):")
            print(oe_info.to_string(index=False))
        else:
            print(f"  {len(no_geo)} ISSNs missing town/state in master.csv:")
            print(no_geo_info.to_string(index=False))

    if has_geo_no_match:
        missing_map = master_geo[master_geo['issn'].isin(has_geo_no_match)][['issn', 'town', 'state']].drop_duplicates('issn')
        print(f"\n  {len(has_geo_no_match)} ISSNs have town/state but no county mapping — add to city_county_mapping.csv:")
        print(missing_map.to_string(index=False))
else:
    print("All analysis ISSNs matched to Census county population.")

# Piecewise linear interpolation between census years (1870-1880, 1880-1890)
df = df.merge(issn_pop[['issn', 'pop_1870', 'pop_1880', 'pop_1890']].drop_duplicates('issn'),
              on='issn', how='left')

def interp_pop(row):
    p70, p80, p90 = row.get('pop_1870'), row.get('pop_1880'), row.get('pop_1890')
    yr = row['year']
    pops = {1870: p70, 1880: p80, 1890: p90}
    known = {y: p for y, p in pops.items() if pd.notna(p)}
    if not known:
        return np.nan
    if len(known) == 1:
        return list(known.values())[0]
    years = sorted(known.keys())
    if yr <= years[0]:
        return known[years[0]]
    if yr >= years[-1]:
        return known[years[-1]]
    for i in range(len(years) - 1):
        if years[i] <= yr <= years[i + 1]:
            y0, y1 = years[i], years[i + 1]
            return known[y0] + (known[y1] - known[y0]) * (yr - y0) / (y1 - y0)

df['county_pop'] = df.apply(interp_pop, axis=1)
df['log_county_pop'] = np.log(df['county_pop'])

print(f"\nCounty population available for {df['log_county_pop'].notna().sum()} / {len(df)} obs")
print(f"Range: {df['county_pop'].min():.0f} - {df['county_pop'].max():.0f}")


  3 ISSNs missing town/state in master.csv:
     issn           newspaper_name town state
2474-2155 the daily state register  NaN   NaN

County population available for 287 / 293 obs
Range: 842 - 1515301


In [12]:
# Save intermediate outputs for downstream notebooks
import os
os.makedirs('intermediate', exist_ok=True)

df.to_csv('intermediate/analysis_sample.csv', index=False)
counts.to_csv('intermediate/sentiment_counts.csv', index=False)
paper_year_rr.to_csv('intermediate/paper_year_rr.csv', index=False)
person_rr.to_csv('intermediate/person_rr.csv', index=False)
print('Saved intermediate files to intermediate/')

Saved intermediate files to intermediate/
